In [1]:
import os
import re
import glob
from pathlib import Path
from typing import Optional

# Extracción de PDFs
import PyPDF2
import pdfplumber
import fitz  # PyMuPDF

# LangChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [25]:
def extract_text_pdfplumber(pdf_path: str) -> list[dict]:
    """Extrae texto página por página usando pdfplumber (bueno para tablas)."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({
                    "text": text,
                    "page": page_num,
                    "total_pages": len(pdf.pages),
                    "source": os.path.basename(pdf_path)
                })
    return pages

def remove_page_numbers(text: str) -> str:
    """Elimina números de página sueltos (líneas que solo contienen un número)."""
    lines = text.split("\n")
    cleaned = [line for line in lines if not re.match(r"^\s*\d{1,4}\s*$", line)]
    return "\n".join(cleaned)


def remove_headers_footers(text: str, patterns: Optional[list[str]] = None) -> str:
    """Elimina encabezados y pies de página basándose en patrones."""
    if patterns is None:
        # Patrones comunes en documentos académicos/técnicos
        patterns = [
            r"^\s*Page\s+\d+\s*(of\s+\d+)?\s*$",       # "Page 5 of 10"
            r"^\s*Página\s+\d+\s*(de\s+\d+)?\s*$",     # "Página 5 de 10"
            r"^\s*-\s*\d+\s*-\s*$",                     # "- 5 -"
            r"^\s*\d+\s*/\s*\d+\s*$",                   # "5/10"
            r"^\s*©.*$",                                 # Líneas de copyright
            r"^\s*All rights reserved\.?\s*$",           # Derechos reservados
            r"^\s*Todos los derechos reservados\.?\s*$",
            r"^\s*Confidential\.?\s*$",                  # Confidencialidad

        ]

    lines = text.split("\n")
    cleaned = []
    for line in lines:
        if not any(re.match(p, line, re.IGNORECASE) for p in patterns):
            cleaned.append(line)
    return "\n".join(cleaned)


def fix_hyphenation(text: str) -> str:
    """Reconecta palabras cortadas por guión al final de línea."""
    # "progra-\nming" -> "programming"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)
    return text


def normalize_whitespace(text: str) -> str:
    """Normaliza espacios en blanco: colapsa múltiples espacios y líneas vacías."""
    # Reemplazar múltiples espacios por uno solo
    text = re.sub(r"[^\S\n]+", " ", text)
    # Reemplazar 3+ líneas vacías por 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Limpiar espacios al inicio/final de cada línea
    lines = [line.strip() for line in text.split("\n")]
    return "\n".join(lines)


def remove_special_characters(text: str) -> str:
    """Elimina caracteres de control y basura Unicode, preservando acentos y ñ."""
    # Eliminar caracteres de control (excepto newlines y tabs)
    print(type(text))
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    # Reemplazar caracteres Unicode problemáticos comunes
    replacements = {
        "\u2018": "'", "\u2019": "'",  # Comillas simples tipográficas
        "\u201c": '"', "\u201d": '"',  # Comillas dobles tipográficas
        "\u2013": "-", "\u2014": "-",  # Guiones em/en
        "\u2026": "...",                 # Elipsis
        "\u00a0": " ",                   # Non-breaking space
        "\ufeff": "",                     # BOM
        "\u200b": "",                     # Zero-width space
        "\uf0b7": "- ",                   # Bullet point (symbol font)
        "\uf0a7": "- ",                   # Otro bullet
    }

    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def join_broken_paragraphs(text: str) -> str:
    """Une líneas que son continuación de un párrafo (no terminan en punto, etc.)."""
    lines = text.split("\n")
    result = []
    for i, line in enumerate(lines):
        if not line.strip():
            result.append(line)
            continue

        # Si la línea anterior no termina en puntuación final y la actual
        # empieza en minúscula, probablemente es continuación del párrafo
        if (result
            and result[-1].strip()
            and not re.search(r"[.!?:;]\s*$", result[-1])
            and re.match(r"^[a-záéíóúüñ]", line.strip())):
            result[-1] = result[-1].rstrip() + " " + line.strip()
        else:
            result.append(line)
    return "\n".join(result)

In [20]:
# Carpeta con los PDFs a procesar
PDF_FOLDER = 'C:/Users/inqui/OneDrive/Desktop/Clases/26-2/Proyecto-LLM/Data/raw'

#PDFs disponibles
pdf_files = sorted(glob.glob(os.path.join(PDF_FOLDER, "*.pdf")))
#pdf_files

'''
for f in pdf_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  - {os.path.basename(f)} ({size_mb:.2f} MB)")
'''

print(pdf_files)

['C:/Users/inqui/OneDrive/Desktop/Clases/26-2/Proyecto-LLM/Data/raw\\LFT.pdf', 'C:/Users/inqui/OneDrive/Desktop/Clases/26-2/Proyecto-LLM/Data/raw\\LFT2.pdf']


In [32]:
def clean_text(
    text: str,
    remove_pages: bool = True,
    remove_hf: bool = True,
    fix_hyphens: bool = True,
    join_paragraphs: bool = True,
    clean_special: bool = True,
    normalize_ws: bool = True,
    custom_header_patterns: Optional[list[str]] = None,
) -> str:
    """Pipeline de limpieza completo. Cada paso se puede activar/desactivar."""

    if clean_special:
        text = remove_special_characters(text)

    if remove_pages:
        text = remove_page_numbers(text)

    if remove_hf:
        text = remove_headers_footers(text, patterns=custom_header_patterns)

    if fix_hyphens:
        text = fix_hyphenation(text)

    if join_paragraphs:
        text = join_broken_paragraphs(text)

    if normalize_ws:
        text = normalize_whitespace(text)

    return text.strip()


# Demostración con texto de ejemplo
sample_dirty_text = extract_text_pdfplumber(pdf_path = pdf_files[0])



print("=== TEXTO ORIGINAL ===")
print(repr(sample_dirty_text[0]['text']))
print()

cleaned = clean_text(sample_dirty_text[0]['text'])
print("=== TEXTO LIMPIO ===")
print(cleaned[:100])

=== TEXTO ORIGINAL ===
'LEY FEDERAL DEL TRABAJO\nCÁMARA DE DIPUTADOS DEL H. CONGRESO DE LA UNIÓN Últimas Reformas DOF 01-05-2026\nSecretaría General\nSecretaría de Servicios Parlamentarios\nLEY FEDERAL DEL TRABAJO\nNueva Ley publicada en el Diario Oficial de la Federación el 1º de abril de 1970\nTEXTO VIGENTE\nÚltimas reformas publicadas 01-05-2026\nAl margen un sello con el Escudo Nacional, que dice: Estados Unidos Mexicanos.- Presidencia de la\nRepública.\nGUSTAVO DIAZ ORDAZ, Presidente Constitucional de los Estados Unidos Mexicanos, a sus\nhabitantes, sabed:\nQue el H. Congreso de la Unión se ha servido dirigirme el siguiente\nD E C R E T O:\nEl Congreso de los Estados Unidos Mexicanos decreta:\nLEY FEDERAL DEL TRABAJO\nTITULO PRIMERO\nPrincipios Generales\nArtículo 1o.- La presente Ley es de observancia general en toda la República y rige las relaciones de\ntrabajo comprendidas en el artículo 123, Apartado A, de la Constitución.\nArtículo 2o.- Las normas del trabajo tienden a conse